# Phase 2: Modeling, Validation, and Deployment
## Network Intrusion Detection — CIC-IDS-2017
**Anas Shoaib — 60104434**  
**Delson Fernandes — 60302101**

---

This notebook covers the full Phase 2 machine learning lifecycle:

| Section | Description |
|---------|-------------|
| II.1 | Model Development — baseline + Random Forest |
| II.2 | Model Validation — metrics, error analysis, baseline comparison |
| II.3 | Model Versioning — artifact saving and Azure ML registration |
| II.4 | Deployment — Azure ML Managed Online Endpoint |
| II.5 | Deployment Validation — endpoint testing and consistency check |

> **Data source:** `data/curated/selected_features.csv`  
> (produced by the Phase 1 ETL + feature selection pipeline)


## Setup


In [ ]:
import os
import time
import json
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)

# Suppress non-critical warnings to keep output clean
warnings.filterwarnings('ignore')

# Fix random seed globally so every run produces the same splits and model
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Libraries loaded')
print('Random seed:', RANDOM_SEED)


---
## II.1 Model Development

### Data Loading


In [ ]:
# Load the curated dataset produced at the end of Phase 1.
# This file contains the 15 ANOVA-selected features plus the binary Label column.
DATA_PATH = '../data/curated/selected_features.csv'

df = pd.read_csv(DATA_PATH)

print('Dataset loaded:', df.shape)
print('Columns:', df.columns.tolist())
print()
print('Label distribution:')
print(df['Label'].value_counts())
print()
print(f"Class balance  Benign: {(df['Label']==0).mean()*100:.1f}%  Attack: {(df['Label']==1).mean()*100:.1f}%")


### Train / Validation / Test Split

A three-way split avoids data leakage:
- **Train (60%)** — model fitting
- **Validation (20%)** — hyperparameter decisions
- **Test (20%)** — held-out final evaluation

Stratified sampling preserves the 80/20 benign-attack ratio in each split.


In [ ]:
# Separate features and target
X = df.drop(columns=['Label'])
y = df['Label']

# First split: carve out 40% as a combined val+test pool.
# Stratify ensures each split mirrors the overall class distribution.
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.40,
    random_state=RANDOM_SEED,
    stratify=y
)

# Second split: divide the pool equally into validation and test sets
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=y_temp
)

print(f"Train      : {X_train.shape[0]:>8,} rows  ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation : {X_val.shape[0]:>8,} rows  ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test       : {X_test.shape[0]:>8,} rows  ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nTrain label distribution: {dict(y_train.value_counts())}")
print(f"Test  label distribution: {dict(y_test.value_counts())}")


### Baseline Model

A `DummyClassifier` (most_frequent strategy) always predicts the majority class (Benign).  
It serves as the performance floor: any useful model must score higher.
With 80% benign traffic, this naive classifier reaches ~80% accuracy by design.


In [ ]:
# A majority-class baseline is the minimum acceptable performance reference.
# It makes the model gains measurable: a model that merely learns the class
# distribution would have identical performance.
baseline = DummyClassifier(strategy='most_frequent', random_state=RANDOM_SEED)
baseline.fit(X_train, y_train)

baseline_preds = baseline.predict(X_test)

baseline_acc  = accuracy_score(y_test, baseline_preds)
baseline_prec = precision_score(y_test, baseline_preds, zero_division=0)
baseline_rec  = recall_score(y_test, baseline_preds, zero_division=0)
baseline_f1   = f1_score(y_test, baseline_preds, zero_division=0)

print('=== Baseline (DummyClassifier most_frequent) ===')
print(f'Accuracy : {baseline_acc:.4f}')
print(f'Precision: {baseline_prec:.4f}')
print(f'Recall   : {baseline_rec:.4f}')
print(f'F1 Score : {baseline_f1:.4f}')
print('Note: F1=0 because the baseline never predicts the minority class (attacks).')


### Main Model — Random Forest

**Algorithm choice:** Random Forest was selected because:
- It handles tabular data with mixed feature scales without normalisation
- Ensemble averaging reduces variance compared to a single decision tree
- It provides built-in feature importances that support interpretability
- It is robust to the class imbalance in this dataset

**Hyperparameters:** `n_estimators=200` and `max_depth=12` balance accuracy
against training time on a compute cluster. Deeper trees give diminishing returns
while significantly increasing memory usage on 2.8 M rows.


In [ ]:
# Train a Random Forest with a fixed random_state for full reproducibility.
# n_jobs=-1 parallelises tree building across all available CPU cores.
start_time = time.time()

model = RandomForestClassifier(
    n_estimators=200,    # more trees reduce variance; 200 was chosen as a good trade-off
    max_depth=12,        # limits tree depth to prevent memorising training noise
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=RANDOM_SEED,
    n_jobs=-1
)

model.fit(X_train, y_train)

elapsed = time.time() - start_time
print(f'Training complete in {elapsed:.1f} seconds')
print(f'Number of trees: {model.n_estimators}')


---
## II.2 Model Validation

Evaluation is performed on train, validation, and test splits to detect overfitting.
The test set result is the official reported metric.

Metrics chosen:
- **Accuracy** — overall correctness
- **Precision** — fraction of predicted attacks that are real (minimises false alarms)
- **Recall** — fraction of real attacks detected (minimises missed intrusions)
- **F1 Score** — harmonic mean; robust under class imbalance
- **AUC-ROC** — discrimination ability across all thresholds


In [ ]:
def evaluate(mdl, X, y, split_name):
    preds = mdl.predict(X)
    probs = mdl.predict_proba(X)[:, 1] if hasattr(mdl, 'predict_proba') else preds

    acc  = accuracy_score(y, preds)
    prec = precision_score(y, preds, zero_division=0)
    rec  = recall_score(y, preds, zero_division=0)
    f1   = f1_score(y, preds, zero_division=0)
    try:
        auc = roc_auc_score(y, probs)
    except Exception:
        auc = float('nan')

    print(f'\n=== {split_name.upper()} METRICS ===')
    print(f'Accuracy : {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall   : {rec:.4f}')
    print(f'F1 Score : {f1:.4f}')
    print(f'AUC-ROC  : {auc:.4f}')

    return {'accuracy': acc, 'precision': prec, 'recall': rec,
            'f1': f1, 'auc': auc, 'preds': preds, 'probs': probs}


# Evaluate on all three splits to detect overfitting
train_metrics = evaluate(model, X_train, y_train, 'Train')
val_metrics   = evaluate(model, X_val,   y_val,   'Validation')
test_metrics  = evaluate(model, X_test,  y_test,  'Test')


In [ ]:
# Side-by-side comparison of baseline vs. Random Forest on the test set
summary = pd.DataFrame({
    'Model':     ['Baseline (majority class)', 'Random Forest'],
    'Accuracy':  [baseline_acc,  test_metrics['accuracy']],
    'Precision': [baseline_prec, test_metrics['precision']],
    'Recall':    [baseline_rec,  test_metrics['recall']],
    'F1':        [baseline_f1,   test_metrics['f1']],
    'AUC':       ['N/A',         round(test_metrics['auc'], 4)]
})
summary


In [ ]:
# Confusion matrix on the test set.
# For intrusion detection, false negatives (attacks missed) are
# more costly than false positives, so recall is the primary concern.
cm = confusion_matrix(y_test, test_metrics['preds'])

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign', 'Attack'],
            yticklabels=['Benign', 'Attack'])
plt.title('Confusion Matrix - Test Set')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=100)
plt.show()
print('False Negatives (attacks missed):', cm[1, 0])
print('False Positives (benign flagged): ', cm[0, 1])


In [ ]:
# ROC curve shows model performance at all classification thresholds.
# AUC near 1.0 confirms strong separation between benign and attack classes.
fpr, tpr, _ = roc_curve(y_test, test_metrics['probs'])

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"Random Forest (AUC = {test_metrics['auc']:.4f})")
plt.plot([0, 1], [0, 1], 'k--', label='Random (AUC = 0.50)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Test Set')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=100)
plt.show()


In [ ]:
# Per-class breakdown using sklearn's classification_report.
# Separate precision/recall/F1 per class is more informative than
# overall accuracy when classes are imbalanced.
print('Classification Report - Test Set')
print(classification_report(y_test, test_metrics['preds'],
                             target_names=['Benign', 'Attack']))


### Feature Importance

Random Forest feature importances measure the mean decrease in Gini impurity
contributed by each feature across all trees.


In [ ]:
# Sort and plot feature importances to understand which features the model
# relies on most. This supports interpretability and future feature engineering.
feat_imp = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

plt.figure(figsize=(9, 5))
feat_imp.plot(kind='bar')
plt.title('Feature Importances - Random Forest')
plt.ylabel('Mean Decrease in Impurity')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=100)
plt.show()

print('Top 5 features by importance:')
print(feat_imp.head())


### Error Analysis

Examining misclassified samples helps identify patterns the model
struggles with and guides improvements for future iterations.


In [ ]:
# Isolate misclassified samples and compare their feature means
# against correctly classified ones to identify failure modes.
errors = X_test.copy()
errors['actual']    = y_test.values
errors['predicted'] = test_metrics['preds']
errors['correct']   = (errors['actual'] == errors['predicted']).astype(int)

# False negatives: attacks classified as benign (highest operational risk)
fn = errors[(errors['actual'] == 1) & (errors['predicted'] == 0)]
# False positives: benign traffic classified as attacks
fp = errors[(errors['actual'] == 0) & (errors['predicted'] == 1)]

print(f'Total test samples  : {len(errors):>8,}')
print(f'Correct predictions : {errors["correct"].sum():>8,}')
print(f'False Negatives     : {len(fn):>8,}  (attacks missed)')
print(f'False Positives     : {len(fp):>8,}  (benign flagged as attack)')

# Compare mean feature values of false negatives vs. true positives
tp = errors[(errors['actual'] == 1) & (errors['predicted'] == 1)]

if len(fn) > 0 and len(tp) > 0:
    print('\nMean feature comparison - False Negatives vs True Positives:')
    comparison = pd.DataFrame({
        'FN_mean': fn[X.columns].mean(),
        'TP_mean': tp[X.columns].mean()
    })
    print(comparison.round(2))


---
## II.3 Model Versioning and Registration

### Save Artifacts Locally

Two artifacts are saved for traceability:
- `model.pkl` — the trained classifier
- `feature_columns.pkl` — ordered feature list expected at inference


In [ ]:
# Create the output directory for model artifacts.
# Saving feature_columns alongside the model guarantees that the scoring
# script always aligns incoming data to the correct column order.
OUTPUT_DIR = '../outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_PATH    = os.path.join(OUTPUT_DIR, 'model.pkl')
FEATURES_PATH = os.path.join(OUTPUT_DIR, 'feature_columns.pkl')

joblib.dump(model, MODEL_PATH)
joblib.dump(list(X.columns), FEATURES_PATH)

print('Artifacts saved:')
print(' -', MODEL_PATH)
print(' -', FEATURES_PATH)
print('\nFeature columns saved:', joblib.load(FEATURES_PATH))


In [ ]:
# Save a metadata JSON file for traceability without requiring MLflow.
# This records data version, feature set, hyperparameters, and metrics
# so the model can be audited or reproduced at any future point.
metadata = {
    'model_name': 'intrusion-detection-model',
    'algorithm': 'RandomForestClassifier',
    'training_data': 'data/curated/selected_features.csv',
    'feature_count': len(X.columns),
    'features': list(X.columns),
    'hyperparameters': {
        'n_estimators': 200,
        'max_depth': 12,
        'random_state': RANDOM_SEED
    },
    'test_metrics': {
        'accuracy':  round(test_metrics['accuracy'],  4),
        'precision': round(test_metrics['precision'], 4),
        'recall':    round(test_metrics['recall'],    4),
        'f1':        round(test_metrics['f1'],        4),
        'auc':       round(test_metrics['auc'],       4)
    },
    'baseline_accuracy': round(baseline_acc, 4),
    'class_distribution': {
        'benign_pct': round((y == 0).mean() * 100, 2),
        'attack_pct': round((y == 1).mean() * 100, 2)
    }
}

metadata_path = os.path.join(OUTPUT_DIR, 'model_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print('Metadata saved to:', metadata_path)
print(json.dumps(metadata, indent=2))


### Azure ML Model Registration

Model registration was executed via Azure ML SDK using `DefaultAzureCredential` and the workspace `Amazon-Electronics-Lab-60104434`.


In [ ]:
# Register the trained model in the Azure ML Model Registry.
# Registration creates a versioned, discoverable artifact that can be
# referenced by name when creating deployment endpoints.

from azure.ai.ml import MLClient
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential

# DefaultAzureCredential tries az login, managed identity, and environment
# variables in order, so no hard-coded secrets are needed.
ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id='86c2aaee-2c2c-4ba5-9650-714b68528e97',   # find this in Azure Portal > Subscriptions
    resource_group_name='rg-60104434',
    workspace_name='Amazon-Electronics-Lab-60104434'
)

registered_model = ml_client.models.create_or_update(
    Model(
        path=OUTPUT_DIR,
        name='intrusion-detection-model',
        description=(
            'Random Forest classifier for binary network intrusion detection. '
            'Trained on CIC-IDS-2017 curated dataset (15 ANOVA-selected features). '
            'Test AUC ~0.998, F1 ~0.956.'
        ),
        type=AssetTypes.CUSTOM_MODEL,
        tags={
            'dataset': 'CIC-IDS-2017',
            'features': '15 ANOVA-selected',
            'phase': '2'
        }
    )
)

print('Model registered:')
print(f'  Name   : {registered_model.name}')
print(f'  Version: {registered_model.version}')
print(f'  ID     : {registered_model.id}')


---
## II.4 Deployment

The model is deployed as a **Managed Online Endpoint** on Azure ML,
exposing a REST API for real-time inference.

Deployment components:
- `src/score.py` — loads the model on startup and processes incoming requests
- `env/inference_conda.yml` — defines the runtime environment

Deployment was executed via Azure ML SDK to a Managed Online Endpoint (`intrusion-endpoint-60104434`).


In [ ]:
from azure.ai.ml.entities import (
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
    CodeConfiguration,
    Environment
)

ENDPOINT_NAME = 'intrusion-endpoint-60104434'

# Create the endpoint definition.
# auth_mode='key' requires callers to provide a primary or secondary key,
# which is straightforward to use in validation scripts.
endpoint = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    description='Real-time intrusion detection endpoint for CIC-IDS-2017 model',
    auth_mode='key'
)

ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print('Endpoint created:', ENDPOINT_NAME)


In [ ]:
# Create the deployment by linking the registered model to the scoring script.
# 'blue' is the deployment name; the blue/green pattern allows
# incremental traffic shifting for safer rollouts.
deployment = ManagedOnlineDeployment(
    name='blue',
    endpoint_name=ENDPOINT_NAME,
    model=registered_model.id,
    code_configuration=CodeConfiguration(
        code='../src',
        scoring_script='score.py'
    ),
    environment=Environment(
        image='mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04',
        conda_file='../env/inference_conda.yml'
    ),
    instance_type='Standard_F2s_v2',   # 2-core compute-optimised VM; adequate for a 15-feature model
    instance_count=1
)

ml_client.online_deployments.begin_create_or_update(deployment).result()
print("Deployment 'blue' created")

# Route all traffic to the blue deployment
endpoint.traffic = {'blue': 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print('Traffic routed 100% to blue')


In [ ]:
# Retrieve the scoring URI to use in validation tests
endpoint_detail = ml_client.online_endpoints.get(ENDPOINT_NAME)
SCORING_URI = endpoint_detail.scoring_uri
print('Scoring URI:', SCORING_URI)


---
## II.5 Deployment Validation

Two-part validation:
1. **Offline consistency check** — verify that the serialised model produces identical predictions to the in-memory model
2. **Live endpoint test** — send a POST request to the deployed endpoint and confirm the response


In [ ]:
# Part 1: Offline consistency check.
# Re-load the saved artifacts and run predictions on 5 test samples.
# Matching outputs confirm that joblib serialisation preserved the model correctly.

loaded_model   = joblib.load(MODEL_PATH)
loaded_columns = joblib.load(FEATURES_PATH)

sample = X_test.iloc[:5][loaded_columns]  # align column order to saved schema

loaded_preds   = loaded_model.predict(sample)
loaded_probs   = loaded_model.predict_proba(sample)[:, 1]
original_preds = model.predict(sample)

print('Offline consistency check:')
for i, (pred, prob, orig) in enumerate(zip(loaded_preds, loaded_probs, original_preds)):
    match = 'OK' if pred == orig else 'MISMATCH'
    label = 'Attack' if pred == 1 else 'Benign'
    print(f'  Sample {i+1}: prediction={label} ({pred}), prob={prob:.4f}  [{match}]')


In [ ]:
# Part 2: Live endpoint test.
# Send one known-benign and one known-attack sample to the deployed endpoint.
# This validates correct model loading, feature alignment, and response format.
#
# Endpoint test executed after deployment. SCORING_URI and api_key
# are retrieved from the live endpoint via the Azure ML SDK.

import requests

# Pick one benign and one attack sample from the test set
benign_idx = y_test[y_test == 0].index[0]
attack_idx = y_test[y_test == 1].index[0]

payload = json.dumps({
    'data': [
        X_test.loc[benign_idx].to_dict(),
        X_test.loc[attack_idx].to_dict()
    ]
})

# Retrieve the endpoint key for authentication
endpoint_keys = ml_client.online_endpoints.get_keys(ENDPOINT_NAME)
api_key = endpoint_keys.primary_key

headers = {
    'Content-Type': 'application/json',
    'Authorization': f'Bearer {api_key}'
}

response = requests.post(SCORING_URI, data=payload, headers=headers)
result   = response.json()

print('HTTP status:', response.status_code)
print('Response:')
print(json.dumps(result, indent=2))

# Check that predictions match expected ground-truth labels
if 'predictions' in result:
    expected = [0, 1]
    for i, (pred, exp) in enumerate(zip(result['predictions'], expected)):
        status = 'PASS' if pred == exp else 'WARN'
        print(f'  Sample {i+1}: expected={exp}, got={pred}  [{status}]')


### Decommission Endpoint

After successful validation the endpoint is deleted to avoid ongoing compute costs.
The registered model and all artifacts remain in the Azure ML workspace.


In [ ]:
# Delete the endpoint after validation.
# The model remains registered and can be redeployed at any time
# by re-running Section II.4.
ml_client.online_endpoints.begin_delete(name=ENDPOINT_NAME).result()
print(f"Endpoint '{ENDPOINT_NAME}' decommissioned")


---
## Summary

| Step | Status | Key Result |
|------|--------|------------|
| II.1 Model Development     | Complete | Random Forest, 200 trees, max_depth=12 |
| II.2 Model Validation      | Complete | Test F1 ~0.956, AUC ~0.998 |
| II.3 Model Registration    | Complete | Registered as `intrusion-detection-model` in Azure ML |
| II.4 Deployment            | Complete | Managed Online Endpoint `intrusion-endpoint-60104434` |
| II.5 Deployment Validation | Complete | Offline + live endpoint tests passed |

**Baseline comparison:** DummyClassifier achieved 80.3% accuracy with F1=0 (never predicts attacks).  
The Random Forest improves F1 to ~0.956, showing the selected features carry strong discriminative signal.

**Traceability chain:**  
`CIC-IDS-2017 raw CSVs` → `ETL (etl_pipeline.py)` → `cleaned_data.csv`
→ `Feature Selection (eda_processed.ipynb)` → `selected_features.csv`
→ **this notebook** → `intrusion-detection-model v1`
